# Thyroid Cancer Recurrence: Random Forest Classification
**Author:** Justin Williams  
**Date:** 11/3/2026

A demonstration of the Random Forest classification model. The notebook compares two evaluation methods: train/test splits and 5-fold cross-validation.

## Contents

1. Background
2. Setup and Constants
3. Load Data
4. Preprocessing
5. Model Pipeline
6. Evaluation
7. Results

## 1. Background

A `RandomForestClassifier` is an ensemble of decision trees. Each tree is trained on a random sample of rows and considers a random selection of features at each split. Each tree votes, and the majority vote is chosen for the final prediction.

| Hyperparameter | Value used | Effect |
|---|---|---|
| `n_estimators` | 10 | Number of trees. More trees give more stable predictions but slower training. |
| `max_depth` | 5 | Maximum depth of each tree. Limits overfitting. |
| `random_state` | 42 | Makes the bootstrap sampling repeatable. |


## 2. Setup and Constants

In [96]:
import numpy as np
import pandas as pd
from sklearn.ensemble import RandomForestClassifier
from sklearn.compose import ColumnTransformer
from sklearn.decomposition import PCA
from sklearn.pipeline import Pipeline
from sklearn.metrics import accuracy_score, confusion_matrix
from sklearn import preprocessing
from sklearn.model_selection import StratifiedKFold, cross_val_predict, cross_val_score, train_test_split

In [97]:
TARGET_COL = "Recurred"
CATEGORICAL_COLS = [
    "Gender",
    "Smoking",
    "Hx Smoking",
    "Hx Radiotherapy",
    "Thyroid Function",
    "Physical Examination",
    "Adenopathy",
    "Pathology",
    "Focality",
    "Risk",
    "T",
    "N",
    "M",
    "Stage",
    "Response"
]

RAND_STATE = 42

TEST_SIZES = [0.2, 0.25, 0.3]

In [98]:
# Formats a confusion matrix (cm) into a Pandas dataframe
def make_cm_frame(cm):
    return pd.DataFrame(
        cm,
        index=["Actual No", "Actual Yes"],
        columns=["Pred No", "Pred Yes"]
    )


### Cross-Validation Setup

`StratifiedKFold` keeps the ratio of target values consistent across each fold.

In [99]:
# Uses StratifiedKFold to keep the ratio of target values consistent across folds
kfold = StratifiedKFold(n_splits=5, shuffle=True, random_state=RAND_STATE)

## 3. Load Data

In [100]:
df = pd.read_csv("../data/processed/Thyroid_Diff.csv")

print(df.shape)

df.head()

(383, 17)


,Age,Gender,Smoking,Hx Smoking,Hx Radiotherapy,Thyroid Function,Physical Examination,Adenopathy,Pathology,Focality,Risk,T,N,M,Stage,Response,Recurred
0,27,F,No,No,No,Euthyroid,Single nodular goiter-left,No,Micropapillary,Uni-Focal,Low,T1a,N0,M0,I,Indeterminate,No
1,34,F,No,Yes,No,Euthyroid,Multinodular goiter,No,Micropapillary,Uni-Focal,Low,T1a,N0,M0,I,Excellent,No
2,30,F,No,No,No,Euthyroid,Single nodular goiter-right,No,Micropapillary,Uni-Focal,Low,T1a,N0,M0,I,Excellent,No
3,62,F,No,No,No,Euthyroid,Single nodular goiter-right,No,Micropapillary,Uni-Focal,Low,T1a,N0,M0,I,Excellent,No
4,62,F,No,No,No,Euthyroid,Multinodular goiter,No,Micropapillary,Multi-Focal,Low,T1a,N0,M0,I,Excellent,No


In [101]:
df.info()

<class 'pandas.DataFrame'>
RangeIndex: 383 entries, 0 to 382
Data columns (total 17 columns):
 #   Column                Non-Null Count  Dtype
---  ------                --------------  -----
 0   Age                   383 non-null    int64
 1   Gender                383 non-null    str  
 2   Smoking               383 non-null    str  
 3   Hx Smoking            383 non-null    str  
 4   Hx Radiotherapy       383 non-null    str  
 5   Thyroid Function      383 non-null    str  
 6   Physical Examination  383 non-null    str  
 7   Adenopathy            383 non-null    str  
 8   Pathology             383 non-null    str  
 9   Focality              383 non-null    str  
 10  Risk                  383 non-null    str  
 11  T                     383 non-null    str  
 12  N                     383 non-null    str  
 13  M                     383 non-null    str  
 14  Stage                 383 non-null    str  
 15  Response              383 non-null    str  
 16  Recurred           

## 4. Preprocessing

`X` holds the features to train on. `y` holds the target to predict. The target column maps Yes/No to 1/0.

In [102]:
X = df.drop(columns=[TARGET_COL])
y = df[TARGET_COL].map({"No": 0, "Yes": 1})

print(y.value_counts())

Recurred
0    275
1    108
Name: count, dtype: int64


### Encoding and Scaling

Columns listed in `CATEGORICAL_COLS` are transformed using One-Hot Encoding. The `Age` column is scaled to have a mean of 0 and a standard deviation of 1. The `ColumnTransformer` is used to avoid data leakage from encoding training and testing data together. It produces the same columns, even if future data is missing any categories.


In [103]:
colTransform = ColumnTransformer(
    transformers=[
        ("cat", preprocessing.OneHotEncoder(handle_unknown="ignore"), CATEGORICAL_COLS),
        ("num", preprocessing.StandardScaler(), ["Age"])
    ],
    verbose_feature_names_out=False
)

## 5. Model Pipeline

In [104]:
model = Pipeline([
    ("trans", colTransform),
    ("clf", RandomForestClassifier(max_depth=5, n_estimators=10, random_state=RAND_STATE))
])

model

,"steps steps: list of tuplesList of (name of step, estimator) tuples that are to be chained insequential order. To be compatible with the scikit-learn API, all stepsmust define `fit`. All non-last steps must also define `transform`. See:ref:`Combining Estimators <combining_estimators>` for more details.","[('trans', ...), ('clf', ...)]"
,"transform_input transform_input: list of str, default=NoneThe names of the :term:`metadata` parameters that should be transformed by thepipeline before passing it to the step consuming it.This enables transforming some input arguments to ``fit`` (other than ``X``)to be transformed by the steps of the pipeline up to the step which requiresthem. Requirement is defined via :ref:`metadata routing <metadata_routing>`.For instance, this can be used to pass a validation set through the pipeline.You can only set this if metadata routing is enabled, which youcan enable using ``sklearn.set_config(enable_metadata_routing=True)``... versionadded:: 1.6",None
,"memory memory: str or object with the joblib.Memory interface, default=NoneUsed to cache the fitted transformers of the pipeline. The last stepwill never be cached, even if it is a transformer. By default, nocaching is performed. If a string is given, it is the path to thecaching directory. Enabling caching triggers a clone of the transformersbefore fitting. Therefore, the transformer instance given to thepipeline cannot be inspected directly. Use the attribute ``named_steps``or ``steps`` to inspect estimators within the pipeline. Caching thetransformers is advantageous when fitting is time consuming. See:ref:`sphx_glr_auto_examples_neighbors_plot_caching_nearest_neighbors.py`for an example on how to enable caching.",None
,"verbose verbose: bool, default=FalseIf True, the time elapsed while fitting each step will be printed as itis completed.",False
,"transformers transformers: list of tuplesList of (name, transformer, columns) tuples specifying thetransformer objects to be applied to subsets of the data.name : str Like in Pipeline and FeatureUnion, this allows the transformer and its parameters to be set using ``set_params`` and searched in grid search.transformer : {'drop', 'passthrough'} or estimator Estimator must support :term:`fit` and :term:`transform`. Special-cased strings 'drop' and 'passthrough' are accepted as well, to indicate to drop the columns or to pass them through untransformed, respectively.columns : str, array-like of str, int, array-like of int, array-like of bool, slice or callable Indexes the data on its second axis. Integers are interpreted as positional columns, while strings can reference DataFrame columns by name. A scalar string or int should be used where ``transformer`` expects X to be a 1d array-like (vector), otherwise a 2d array will be passed to the transformer. A callable is passed the input data `X` and can return any of the above. To select multiple columns by name or dtype, you can use :obj:`make_column_selector`.","[('cat', ...), ('num', ...)]"
,"verbose_feature_names_out verbose_feature_names_out: bool, str or Callable[[str, str], str], default=True- If True, :meth:`ColumnTransformer.get_feature_names_out` will prefix all feature names with the name of the transformer that generated that feature. It is equivalent to setting `verbose_feature_names_out=""{transformer_name}__{feature_name}""`.- If False, :meth:`ColumnTransformer.get_feature_names_out` will not prefix any feature names and will error if feature names are not unique.- If ``Callable[[str, str], str]``, :meth:`ColumnTransformer.get_feature_names_out` will rename all the features using the name of the transformer. The first argument of the callable is the transformer name and the second argument is the feature name. The returned string will be the new feature name.- If ``str``, it must be a string ready for formatting. The given string will be formatted using two field names: ``transformer_name`` and ``feature_name``. e.g. ``""{feature_name}__{trans

## 6. Evaluation

### Train-Test Splits

In [105]:
results = []
split_cms = {}

for size in TEST_SIZES:
    X_train, X_test, y_train, y_test = train_test_split(
        X, y, test_size=size, stratify=y, random_state=RAND_STATE
    )
    model.fit(X_train, y_train)
    y_pred = model.predict(X_test)

    results.append({
        "Method": f"Train/test split ({size})",
        "Accuracy": accuracy_score(y_test, y_pred)
    })
    split_cms[size] = confusion_matrix(y_test, y_pred)

### 5-Fold Cross-Validation

In [106]:
cv_scores = cross_val_score(model, X, y, cv=kfold, scoring="accuracy")
cv_pred = cross_val_predict(model, X, y, cv=kfold)
cv_cm = confusion_matrix(y, cv_pred)

results.append({
    "Method": "5-fold cross-validation (mean)",
    "Accuracy": np.mean(cv_scores)
})

## 7. Results

### Accuracy Comparison

In [107]:
results_df = pd.DataFrame(results)

print(results_df.to_string(index=False, float_format="{:.4f}".format))

                        Method  Accuracy
        Train/test split (0.2)    0.9610
       Train/test split (0.25)    0.9688
        Train/test split (0.3)    0.9391
5-fold cross-validation (mean)    0.9374


In [108]:
print("CV accuracy per fold:", np.round(cv_scores, 4))
print("CV accuracy std dev:", np.round(np.std(cv_scores), 4))

CV accuracy per fold: [0.9481 0.9221 0.9091 0.9737 0.9342]
CV accuracy std dev: 0.0223


### Confusion Matrices

The rows represent the actual classes, and the columns represent predicted classes. All correct predictions lie on the diagonal:

| | Pred No | Pred Yes |
|---|---|---|
| **Actual No** | True negatives (TN) | False positives (FP) |
| **Actual Yes** | False negatives (FN) | True positives (TP) |

Train/Test Split matrices count the test set while the Cross-Validation matrix counts the entire dataset.

In [109]:
for size, cm in split_cms.items():
    print(f"Confusion matrix (test_size={size}):")
    display(make_cm_frame(cm))

Confusion matrix (test_size=0.2):


,Pred No,Pred Yes
Actual No,55,0
Actual Yes,3,19


Confusion matrix (test_size=0.25):


,Pred No,Pred Yes
Actual No,69,0
Actual Yes,3,24


Confusion matrix (test_size=0.3):


,Pred No,Pred Yes
Actual No,82,1
Actual Yes,6,26


In [110]:
print("Confusion matrix (5-fold CV):")
display(make_cm_frame(cv_cm))

Confusion matrix (5-fold CV):


,Pred No,Pred Yes
Actual No,267,8
Actual Yes,16,92
